In [ ]:
import csv
import datetime  # noqa: F401
import json
import logging
import os  # noqa: F401
import pathlib
import random
import sqlite3  # noqa: F401
import sys
import time  # noqa: F401
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# ── colour helpers ────────────────────────────────────────────────
GRN = "\033[92m"
BLU = "\033[94m"
YLW = "\033[93m"
RED = "\033[91m"
CYN = "\033[96m"
DIM = "\033[2m"
RST = "\033[0m"
BLD = "\033[1m"


def section(n, title):
    print(f"\n{BLU}{'═' * 60}{RST}")
    print(f"{BLD}  SECTION {n}  —  {title}{RST}")
    print(f"{BLU}{'═' * 60}{RST}\n")


def note(text):
    print(f"{YLW}  ▸ {text}{RST}")


def good(text):
    print(f"{GRN}  ✓ {text}{RST}")


def show(k, v):
    print(f"  {DIM}{k}:{RST}  {v}")


try:
    import pandas as pd

    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

try:
    import pyarrow as pa  # noqa: F401

    HAS_PYARROW = True
except ImportError:
    HAS_PYARROW = False

try:
    from sqlalchemy import create_engine, text  # noqa: F401

    HAS_SQLALCHEMY = True
except ImportError:
    HAS_SQLALCHEMY = False

In [36]:
# =====================================================================
# SECTION 1 — ENVIRONMENT CHECK
# =====================================================================
section(1, "Environment & Project Layout")

"""
[INSTRUCTOR]
Keep this on screen while students join the Meet call.
It validates their local environment in one shot, shows which
packages are missing, and introduces the professional project
directory structure every pipeline should follow.
"""

print(f"  Python : {sys.version.split()[0]}")
print(f"  CWD    : {pathlib.Path.cwd()}")
print()

packages = {
    "pandas": ("Data manipulation — DataFrames", HAS_PANDAS),
    "pyarrow": ("Parquet read/write (pip install pyarrow)", HAS_PYARROW),
    "sqlalchemy": ("Database abstraction layer", HAS_SQLALCHEMY),
}
print(f"  {'Package':<14} {'Status':<12} Purpose")
print(f"  {'─' * 14} {'─' * 12} {'─' * 35}")
for pkg, (purpose, ok) in packages.items():
    status = f"{GRN}installed{RST}" if ok else f"{RED}MISSING{RST}"
    print(f"  {pkg:<14} {status:<20} {purpose}")

print()
note("Professional project layout — use this structure for every pipeline:")
print("""
  my_pipeline/
  ├── data/
  │   ├── raw/          ← source data, NEVER modified after landing
  │   ├── processed/    ← cleaned / transformed output
  │   └── archive/      ← versioned backups (or object storage)
  ├── src/
  │   ├── extract.py    ← ingestion logic
  │   ├── transform.py  ← business logic / cleaning
  │   ├── load.py       ← write to destination
  │   └── utils.py      ← shared helpers
  ├── tests/            ← unit tests (pytest)
  ├── logs/             ← application logs
  ├── .env              ← secrets — NEVER commit to git!
  ├── requirements.txt  ← pinned dependencies
  └── README.md         ← what this does and how to run it
""")

# Create demo folders
ROOT = pathlib.Path("demo_data")
for sub in ["data/raw", "data/processed", "logs"]:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
    print(f"Path is {ROOT}")
good(f"Created: {ROOT.resolve()}")


════════════════════════════════════════════════════════════
  SECTION 1  —  Environment & Project Layout
════════════════════════════════════════════════════════════

  Python : 3.12.12
  CWD    : /Users/mac/engineering/de-spec-ability014/py-project

  Package        Status       Purpose
  ────────────── ──────────── ───────────────────────────────────
  pandas         installed   Data manipulation — DataFrames
  pyarrow        installed   Parquet read/write (pip install pyarrow)
  sqlalchemy     installed   Database abstraction layer

  ▸ Professional project layout — use this structure for every pipeline:

  my_pipeline/
  ├── data/
  │   ├── raw/          ← source data, NEVER modified after landing
  │   ├── processed/    ← cleaned / transformed output
  │   └── archive/      ← versioned backups (or object storage)
  ├── src/
  │   ├── extract.py    ← ingestion logic
  │   ├── transform.py  ← business logic / cleaning
  │   ├── load.py       ← write to destination
  │   └── utils.

In [12]:
# =====================================================================
# SECTION 2 — LOGGING & ERROR HANDLING (Lesson 1.3)
# =====================================================================
section(2, "Professional Logging & Error Handling (Lesson 1.3)")

"""
[INSTRUCTOR]
This is a non-negotiable professional habit. A pipeline that silently
fails at 3am, leaving no trace, is the most dangerous pipeline.
Contrasting print() vs structured logging makes the production value
immediately obvious. Walk through:
  — why timestamps matter (when did this fail?)
  — why log levels matter (INFO vs WARNING vs ERROR)
  — the catch → log → re-raise pattern (show the log output)
"""

log_file = ROOT / "logs" / "pipeline.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(log_file),  # persists to disk
        logging.StreamHandler(sys.stdout),  # also shows in terminal
    ],
)
logger = logging.getLogger("de_demo")

note("Structured logging — compare to print():")
print()
logger.info("Pipeline started — Python %s", sys.version.split()[0])
logger.warning("This is a WARNING — system still running, but worth noting")

print()
note("Log is also written to disk — inspect it after the session:")
note(f"  cat {log_file}")
print()

2026-08-22 13:47:19 | INFO     | de_demo | Pipeline started — Python 3.12.12
2026-08-22 13:47:19 | WARNING  | de_demo | This is a WARNING — system still running, but worth noting



════════════════════════════════════════════════════════════
  SECTION 2  —  Professional Logging & Error Handling (Lesson 1.3)
════════════════════════════════════════════════════════════

  ▸ Structured logging — compare to print():


  ▸ Log is also written to disk — inspect it after the session:
  ▸   cat demo_data/logs/pipeline.log



In [ ]:
# =====================================================================
# SECTION 3 — WORKING WITH FILES (Lesson 1.3)
# =====================================================================
section(3, "Working with Files — pathlib & pandas (Lesson 1.3)")

"""
[INSTRUCTOR]
pathlib is the modern standard (Python 3.4+) — avoid os.path.
Show the attribute API: .name, .stem, .suffix, .parent, .stat().
Then load the CSV with explicit dtypes — stress that inferring dtypes
reads the file twice and produces wrong types for dates and categoricals.
"""

# ── Generate a synthetic taxi dataset (no download needed) ─────────
print("  Generating sample NYC Taxi dataset (10,000 rows)...")

random.seed(42)
ZONES = {1: "JFK Airport", 132: "LaGuardia", 161: "Midtown", 230: "Times Square"}
ZONE_IDS = list(ZONES.keys())
PAYMENT = ["credit_card", "cash", "no_charge", "dispute"]

rows = []
for i in range(10_000):
    dist = round(random.uniform(0.5, 25.0), 2)
    fare = round(max(2.50, 2.50 + dist * 2.50 + random.gauss(0, 3)), 2)
    tip = round(fare * random.uniform(0, 0.25), 2) if random.random() > 0.3 else 0.0
    hr = random.randint(0, 23)
    day = random.randint(1, 28)
    rows.append(
        {
            "trip_id": i + 1,
            "vendor_id": random.choice([1, 2]),
            "pickup_datetime": f"2024-01-{day:02d} {hr:02d}:{random.randint(0, 59):02d}:00",
            "dropoff_datetime": f"2024-01-{day:02d} {(hr + random.randint(0, 2)) % 24:02d}:{random.randint(0, 59):02d}:00",
            "pickup_zone_id": random.choice(ZONE_IDS),
            "dropoff_zone_id": random.choice(ZONE_IDS),
            "passenger_count": random.randint(1, 4),
            "trip_distance": dist,
            "fare_amount": fare,
            "tip_amount": tip,
            "total_amount": round(fare + tip + 0.50, 2),
            "payment_type": random.choice(PAYMENT),
        }
    )


raw_csv = ROOT / "data" / "raw" / "yellow_taxi_jan2024_sample.csv"

# Write via stdlib csv (show the manual way first, then pandas)
with open(raw_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

good(f"Dataset written: {raw_csv.name}  ({len(rows):,} rows)")
print()


════════════════════════════════════════════════════════════
  SECTION 3  —  Working with Files — pathlib & pandas (Lesson 1.3)
════════════════════════════════════════════════════════════

  Generating sample NYC Taxi dataset (10,000 rows)...
  ✓ Dataset written: yellow_taxi_jan2024_sample.csv  (10,000 rows)



In [29]:
# Save data to test_data as json to directory for testing purposes

data = [
    {"name": "Alice", "age": 25, "city": "New York", "date": "2015-01-13"},
    {"name": "Bob", "age": 30, "city": "Los Angeles", "date": "2010-01-13"},
    {"name": "Charlie", "age": 35, "city": "Chicago", "date": "2011-01-13"},
    {"name": "Joe", "age": 28, "city": "Houston", "date": "2015-01-13"},
    {"name": "Emma", "age": 22, "city": "Phoenix", "date": "2015-01-13"},
]
TEST_ROOT = pathlib.Path("test_data")
json_file = TEST_ROOT / "test_data.json"
with open(json_file, "w") as f:
    json.dump(data, f, indent=4)

In [30]:
# Save data to test_data as parquet to directory for testing purposes

test_df = pd.DataFrame(data)

TEST_ROOT = pathlib.Path("test_data")
TEST_ROOT.mkdir(parents=True, exist_ok=True)
test_df.to_parquet(TEST_ROOT / "test_data.parquet", index=False)

In [31]:
# Save data to test_data as csv to directory for testing purposes

test_df.to_csv(TEST_ROOT / "test_data.csv", index=False)

In [ ]:
transformation = {
    "bank_name": "uppercase",
    "birth_date": ("type", "date", "yyyy-mm-dd"),
    "amount": ("type", "int64"),
    "first_name": "uppercase",
}

for col, transform in transformation:
    if transform[0] == "type":
        print(1)

In [32]:
test_df

,name,age,city,date
0,Alice,25,New York,2015-01-13
1,Bob,30,Los Angeles,2010-01-13
2,Charlie,35,Chicago,2011-01-13
3,Joe,28,Houston,2015-01-13
4,Emma,22,Phoenix,2015-01-13


In [33]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    5 non-null      str  
 1   age     5 non-null      int64
 2   city    5 non-null      str  
 3   date    5 non-null      str  
dtypes: int64(1), str(3)
memory usage: 404.0 bytes


In [ ]:
pd.to_datetime(test_df["date"], format="%Y-%m-%d")

0   2015-01-13
1   2010-01-13
2   2011-01-13
3   2015-01-13
4   2015-01-13
Name: date, dtype: datetime64[us]